# stock-bqml POC Pipeline\nBigQuery ML で株価の翌日リターン予測を体験する。\n\n- Silver: テクニカル指標生成\n- Gold: XGBoost 学習・予測・解釈

In [ ]:
# G CP 設定\nPROJECT_ID = 'your-gcp-project'\nREGION = 'US'\n\nfrom google.colab import auth\nauth.authenticate_user()\n!gcloud config set project {PROJECT_ID}

## 1. Silver 特徴量生成

In [ ]:
# features_daily.sql を実行\n!bq query --use_legacy_sql=false \\n  --replace \\n  --destination_table={PROJECT_ID}:stock_silver.features_daily \\n  < silver/features_daily.sql

## 2. Gold モデル学習

In [ ]:
!bq query --use_legacy_sql=false < gold/train_xgb.sql

## 3. 予測

In [ ]:
!bq query --use_legacy_sql=false < gold/predict.sql

## 4. 解釈（特徴量重要度）

In [ ]:
import pandas as pd\ndf = pd.read_gbq('SELECT * FROM ML.GLOBAL_EXPLAIN(MODEL `stock_gold.xgb_predictor`)', project_id=PROJECT_ID)\ndf.head(10)

## Appendix: 結果可視化

In [ ]:
pred = pd.read_gbq('SELECT * FROM `stock_gold.predictions` ORDER BY date DESC LIMIT 100', project_id=PROJECT_ID)\npred.plot(x='date', y='predicted_next_day_return', figsize=(12,4))

## 5. 戦略別バックテスト精度比較
4戦略モデル (momentum / breakout / volume-confirm / reversal) の精度を横並び評価。
実行に数分かかる場合あり。

In [ ]:
# 戦略別モデル学習 + バックテスト一括実行
!bq query --use_legacy_sql=false < gold/strategies.sql
!bq query --use_legacy_sql=false < gold/backtest_accuracy.sql

In [ ]:
# モデル別精度サマリー
df_summary = pd.read_gbq('SELECT * FROM `stock_gold.backtest_model_summary`', project_id=PROJECT_ID)
df_summary.T

In [ ]:
# シグナル別パフォーマンス
df_signal = pd.read_gbq('''SELECT * FROM `stock_gold.backtest_signal_performance`
ORDER BY CASE signal
  WHEN 'STRONG_MOMENTUM' THEN 1 WHEN 'BREAKOUT_CONFIRM' THEN 2
  WHEN 'REVERSAL_BUY' THEN 3 WHEN 'REVERSAL_SELL' THEN 4 ELSE 5 END''', project_id=PROJECT_ID)
df_signal

In [ ]:
# 累積リターン曲線
df_cumul = pd.read_gbq('''SELECT date, momentum_cumul, breakout_cumul,
  volume_confirm_cumul, reversal_cumul, signal_cumul
FROM `stock_gold.backtest_cumulative_returns`
ORDER BY date''', project_id=PROJECT_ID)

ax = df_cumul.plot(x='date', figsize=(14, 6), linewidth=1.5,
  title='Backtest: Cumulative Returns by Strategy')
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
from matplotlib import pyplot as plt
plt.ylabel('Cumulative Return (1.0 = no change)')
plt.grid(alpha=0.3)

In [ ]:
# 方向一致率 & 年化Sharpe比 比較バーチャート
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左: 方向一致率
dir_cols = ['momentum_dir_accuracy', 'breakout_dir_accuracy',
            'volume_confirm_dir_accuracy', 'reversal_dir_accuracy']
dir_vals = [float(df_summary[c].iloc[0]) for c in dir_cols]
labels = ['Momentum', 'Breakout', 'Volume-Conf', 'Reversal']
colors = ['#2196F3', '#FF9800', '#4CAF50', '#E91E63']
ax1.bar(labels, dir_vals, color=colors, alpha=0.8)
ax1.set_ylabel('Directional Accuracy')
ax1.set_title('Model Directional Accuracy')
ax1.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='Baseline (50%)')
ax1.legend()
ax1.set_ylim(0, 1)
for i, v in enumerate(dir_vals):
    ax1.text(i, v + 0.01, f'{v:.1%}', ha='center', fontsize=10)

# 右: シグナル別Sharpe比
sharpe_vals = df_signal['sharpe_ratio_annualized'].values
sig_labels = df_signal['signal'].values
sig_colors = ['#1565C0', '#E65100', '#2E7D32', '#C62828', '#757575']
ax2.bar(range(len(sig_labels)), sharpe_vals, color=sig_colors[:len(sig_labels)], alpha=0.8)
ax2.set_xticks(range(len(sig_labels)))
ax2.set_xticklabels(sig_labels, rotation=15, ha='right')
ax2.set_ylabel('Annualized Sharpe Ratio')
ax2.set_title('Signal Performance (Sharpe)')
ax2.axhline(0, color='gray', linestyle='--', alpha=0.5)
for i, v in enumerate(sharpe_vals):
    ax2.text(i, v + 0.05, f'{v:.2f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()